# Alaska Wildfire Prediction — Sentinel-2 Pipeline Demo

This notebook walks through the complete Sentinel-2 preprocessing pipeline step by step.

## Pipeline Steps
1. Download Sentinel-2 data (or use synthetic demo data)
2. Apply cloud masking using QA60 band
3. Extract spectral bands (B2, B3, B4, B8, B11, B12)
4. Compute vegetation indices (NDVI, NBR, NDMI)
5. Compute fire risk score
6. Normalize and patch for model input

**Note:** Section 1 requires real Sentinel-2 data. Sections 2-6 use synthetic data so you can run the full pipeline without downloading anything.

## 0. Setup — Install Dependencies

In [ ]:
# Run this cell once to install dependencies
# !pip install rasterio numpy matplotlib sentinelsat earthengine-api geemap

import sys
sys.path.insert(0, '..')   # Add project root to path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

print('Setup complete!')

## 1. Download Sentinel-2 Data

**Skip this section if you don't have credentials yet.**  
Use Section 2 instead, which generates synthetic data for testing.

In [ ]:
# OPTION A: Google Earth Engine (Recommended — easier setup)
# from data.sentinel2.downloader import download_via_gee
# tif_path = download_via_gee(
#     region='interior',
#     start_date='2023-06-01',
#     end_date='2023-08-31',
#     cloud_cover_max=30
# )

# OPTION B: Copernicus Hub
# from data.sentinel2.downloader import download_via_sentinelsat
# tif_path = download_via_sentinelsat('your_username', 'your_password')

print('Skipping download — using synthetic data in next section.')

## 2. Create Synthetic Test Data

We simulate a realistic Sentinel-2 scene with:
- Dense forest regions (high NDVI)
- Open tundra/grassland (medium NDVI)
- Water bodies (negative NDVI)
- A simulated burned area (low NBR)
- Some cloud patches (will be masked)

In [ ]:
np.random.seed(42)
H, W = 512, 512

# --- Create land cover zones ---
land_cover = np.zeros((H, W))     # 0 = background

# Forest (high NIR, low SWIR)
land_cover[50:200, 50:200] = 1    # Dense boreal forest
land_cover[250:400, 300:480] = 1  # Another forest patch

# Tundra/shrub (medium NIR)
land_cover[200:300, 50:300] = 2

# Water body (low reflectance everywhere)
land_cover[400:480, 100:250] = 3

# Burned area (low NIR, high SWIR)
land_cover[100:200, 300:450] = 4  # Simulates post-fire area

# --- Generate band values based on land cover ---
def make_band(base_values, noise=0.02):
    """Create a band with different values per land cover class."""
    band = np.zeros((H, W))
    for lc_class, value in enumerate(base_values):
        band[land_cover == lc_class] = value
    band += np.random.normal(0, noise, (H, W))
    return np.clip(band, 0, 1)

# Values: [background, forest, tundra, water, burned]
# Reflectance values roughly in [0,1] range (real data is 0-10000, but normalized here)
synthetic_bands = {
    'B2_blue'  : make_band([0.05, 0.03, 0.04, 0.08, 0.06]),
    'B3_green' : make_band([0.06, 0.05, 0.07, 0.06, 0.05]),
    'B4_red'   : make_band([0.07, 0.04, 0.08, 0.04, 0.10]),
    'B8_nir'   : make_band([0.20, 0.45, 0.30, 0.05, 0.12]),   # NIR: forest > tundra > burned > water
    'B11_swir1': make_band([0.10, 0.08, 0.15, 0.02, 0.25]),   # SWIR1: burned high
    'B12_swir2': make_band([0.08, 0.05, 0.12, 0.01, 0.22]),   # SWIR2: burned very high
}

# --- Simulate QA60 cloud mask ---
qa60 = np.zeros((H, W), dtype=np.int32)
qa60[310:370, 150:280] = 1024   # Cloud patch (bit 10 set)
qa60[20:50, 400:480]   = 2048   # Cirrus patch (bit 11 set)

print(f'Synthetic scene created: {H}x{W} pixels')
print(f'Land cover classes: forest, tundra, water, burned area')
print(f'Cloud patches simulated in 2 regions')

## 3. Apply Cloud Masking

In [ ]:
from data.sentinel2.cloud_mask import create_cloud_mask, apply_cloud_mask, visualize_cloud_mask

# Create cloud mask from QA60 band
clear_mask = create_cloud_mask(qa60)

# Stack bands into array for masking
bands_array = np.stack([
    synthetic_bands['B2_blue'],
    synthetic_bands['B3_green'],
    synthetic_bands['B4_red'],
    synthetic_bands['B8_nir'],
    synthetic_bands['B11_swir1'],
    synthetic_bands['B12_swir2'],
], axis=0)   # Shape: (6, H, W)

# Apply mask
masked_bands_array = apply_cloud_mask(bands_array, clear_mask)

# Report cloud coverage
cloud_pct = (1 - np.mean(clear_mask)) * 100
print(f'Cloud coverage: {cloud_pct:.1f}%')
print(f'Masked bands shape: {masked_bands_array.shape}')

# Visualize
visualize_cloud_mask(masked_bands_array, clear_mask)

## 4. Extract Bands and Compute Vegetation Indices

In [ ]:
from data.sentinel2.band_extractor import extract_bands, compute_all_indices, compute_fire_risk_score, visualize_indices

# Extract named bands from masked array
bands = extract_bands(masked_bands_array)

# Compute NDVI, NBR, NDMI
indices = compute_all_indices(bands)

# Compute fire risk score
risk = compute_fire_risk_score(indices)

# Visualize everything
visualize_indices(bands, indices, risk)

## 5. Normalize and Extract Patches for Model

In [ ]:
from data.sentinel2.normalizer import prepare_for_model

# Prepare bands for model input
patches, positions, norm_stats = prepare_for_model(
    masked_bands_array,
    method='percentile',
    patch_size=64,
    stride=32
)

print(f'\n=== Ready for Model Training ===')
print(f'  Input patches : {patches.shape}  (n_patches, bands, H, W)')
print(f'  Value range   : [{patches.min():.3f}, {patches.max():.3f}]')
print(f'  NaN fraction  : {np.mean(np.isnan(patches)):.3f}')
print(f'  Patch positions stored: {len(positions)}')

## 6. Summary: What We Built

| Step | Module | What it does |
|------|--------|--------------|
| Download | `downloader.py` | Fetch Sentinel-2 tiles from GEE or Copernicus |
| Cloud mask | `cloud_mask.py` | Remove cloudy pixels using QA60 bitmask |
| Band extraction | `band_extractor.py` | Extract B2-B12, compute NDVI/NBR/NDMI |
| Normalization | `normalizer.py` | Scale values to [0,1], split into patches |

### Next Steps
- Integrate Sentinel-1 SAR data (soil moisture, vegetation density)
- Integrate ERA5/NOAA weather data (temperature, wind, humidity)
- Train CNN-LSTM model on this preprocessed data
- Build GIS dashboard to visualize predictions